## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: https://planetarycomputer.microsoft.com/dataset/terraclimate#overview 

In [2]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os
import time

import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()

# Chunked/resumable config
CHUNK_SIZE = 200
RESUME = True
CHUNK_RETRIES = 3
RETRY_SLEEP_S = 5
CHECKPOINT_DIR = os.path.join(os.getcwd(), "tc_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
SAVE_EVERY_ROWS = 50

# Cache filtered TerraClimate grids per variable
FILTER_CACHE_DIR = os.path.join(os.getcwd(), "tc_filtered")
os.makedirs(FILTER_CACHE_DIR, exist_ok=True)


<h2>Extracting TerraClimate Data Using API Calls</h2> <p align="justify"> The API-based method allows us to efficiently access <b>TerraClimate</b> data for specific regions and time periods through the <a href="https://planetarycomputer.microsoft.com/">Microsoft Planetary Computer</a>, ensuring scalability and reproducibility of the process. </p> <p align="justify"> Through the API, we can extract climate variables such as <b>Potential Evapotranspiration (PET)</b>, which represents the atmospheric demand for water. This variable provides critical insights into surface moisture balance and helps improve the accuracy of water quality modeling. </p> <p align="justify"> This approach ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis. </p>



<h3>Loading and Mapping TerraClimate Data:</h3>

<p>This section demonstrates how <b>TerraClimate climate variables</b>, such as <b>Potential Evapotranspiration (PET)</b>, are loaded and mapped to sampling locations:</p>

<ul>
  <li>The <b>load_terraclimate_dataset</b> function opens the TerraClimate Zarr/NetCDF dataset from the Microsoft Planetary Computer, handling storage options automatically.</li>
  <li>The <b>filterg</b> function filters the dataset for the desired time range (2011–2015) and spatial extent corresponding to the study region. The resulting data is converted to a pandas DataFrame with standardized column names.</li>
  <li>The <b>assign_nearest_climate</b> function maps each sampling location to its <b>nearest TerraClimate grid point</b> using a KD-tree and assigns the climate variable values corresponding to the closest time stamp.</li>
</ul>

<p>This workflow ensures efficient, reproducible retrieval of climate variables, while allowing participants to work with pre-extracted CSV files for faster benchmarking and analysis.</p>


In [3]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [4]:
# Override filterg with on-disk cache

def filterg(ds, var):
    cache_path = os.path.join(FILTER_CACHE_DIR, f"{var}_filtered.csv")
    if os.path.exists(cache_path):
        return pd.read_csv(cache_path)

    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    df_var_final.to_csv(cache_path, index=False)
    return df_var_final


In [5]:
# Override assign_nearest_climate_chunked to resume within a chunk

def assign_nearest_climate_chunked(sa_df, climate_df, var_name, checkpoint_path):
    """Row-level checkpointing to resume mid-chunk."""
    sa_df = sa_df.reset_index(drop=True)

    # Precompute nearest grid point for each sample
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    start_idx = 0
    if RESUME and os.path.exists(checkpoint_path):
        existing = pd.read_csv(checkpoint_path)
        start_idx = len(existing)
        print(f"Resuming {var_name} from row {start_idx}")

    total = len(sa_df)
    buffer = []

    def _flush_buffer():
        nonlocal buffer
        if not buffer:
            return
        df_out = pd.DataFrame({var_name: buffer})
        write_header = not os.path.exists(checkpoint_path) or start_idx == 0 and os.path.getsize(checkpoint_path) == 0
        df_out.to_csv(checkpoint_path, mode='a', header=write_header, index=False)
        buffer = []

    try:
        for i in range(start_idx, total):
            sample_date = sa_df.loc[i, 'Sample Date']
            nearest_lat = sa_df.loc[i, 'nearest_lat']
            nearest_lon = sa_df.loc[i, 'nearest_lon']

            subset = climate_df[
                (climate_df['Latitude'] == nearest_lat) &
                (climate_df['Longitude'] == nearest_lon)
            ]

            if subset.empty:
                buffer.append(np.nan)
            else:
                nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
                buffer.append(subset.loc[nearest_idx, var_name])

            # Save every N rows
            if (i + 1) % SAVE_EVERY_ROWS == 0:
                _flush_buffer()

        _flush_buffer()
    except Exception as e:
        # Save progress before raising
        _flush_buffer()
        raise

    return pd.read_csv(checkpoint_path)


In [6]:
# --- Chunked + resumable mapping ---
def assign_nearest_climate_chunked(sa_df, climate_df, var_name, checkpoint_path):
    """Chunked + resumable mapping using checkpoint CSV."""
    sa_df = sa_df.reset_index(drop=True)

    # Precompute nearest grid point for each sample
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    start_idx = 0
    if RESUME and os.path.exists(checkpoint_path):
        existing = pd.read_csv(checkpoint_path)
        start_idx = len(existing)
        print(f"Resuming {var_name} from row {start_idx}")
    else:
        existing = None

    results = []
    if existing is not None and start_idx > 0:
        results.append(existing)

    total = len(sa_df)
    for chunk_start in range(start_idx, total, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, total)

        for attempt in range(1, CHUNK_RETRIES + 1):
            try:
                chunk_vals = []
                for i in range(chunk_start, chunk_end):
                    sample_date = sa_df.loc[i, 'Sample Date']
                    nearest_lat = sa_df.loc[i, 'nearest_lat']
                    nearest_lon = sa_df.loc[i, 'nearest_lon']

                    subset = climate_df[
                        (climate_df['Latitude'] == nearest_lat) &
                        (climate_df['Longitude'] == nearest_lon)
                    ]

                    if subset.empty:
                        chunk_vals.append(np.nan)
                        continue

                    nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
                    chunk_vals.append(subset.loc[nearest_idx, var_name])

                chunk_df = pd.DataFrame({var_name: chunk_vals})
                results.append(chunk_df)

                # Save checkpoint
                pd.concat(results, ignore_index=True).to_csv(checkpoint_path, index=False)
                print(f"{var_name}: saved rows {chunk_start}..{chunk_end - 1}")
                break
            except Exception as e:
                if attempt == CHUNK_RETRIES:
                    raise
                print(f"{var_name} chunk {chunk_start}-{chunk_end} failed (attempt {attempt}): {e}")
                time.sleep(RETRY_SLEEP_S * attempt)

    return pd.concat(results, ignore_index=True)


In [7]:
# --- Filtering function (kept identical) ---
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final


In [8]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

### Extracting features for the training dataset

In [9]:
# Load water quality training dataset (repo root)
Water_Quality_df = pd.read_csv('Datasets_Provided/water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [10]:
Water_Quality_df.shape

(9319, 6)

In [11]:
# Load TerraClimate dataset, filter (time,region,parameter), filter for nearest parameter values
vars_to_extract = [
    "pet", "aet", "def", "q", "ppt", "soil", "swe",
    "srad", "tmax", "tmin", "vap", "vpd", "ws", "pdsi",
]

ds = load_terraclimate_dataset()

feature_dfs = []
for var in vars_to_extract:
    while True:
        try:
            tc_parameter = filterg(ds, var)
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"train_{var}.csv")
            feature_dfs.append(assign_nearest_climate_chunked(Water_Quality_df, tc_parameter, var, ckpt_path))
            break
        except Exception as e:
            err_text = str(e)
            # Refresh signed URL if token expires mid-run
            if "AuthenticationFailed" in err_text or "Signature not valid" in err_text:
                print(f"Token expired while processing {var}, refreshing dataset...")
                time.sleep(5)
                ds = load_terraclimate_dataset()
                continue
            # Retry on transient socket timeouts
            if "Timeout" in err_text or "SocketTimeoutError" in err_text or "ServiceResponseTimeoutError" in err_text:
                print(f"Timeout while processing {var}, retrying after backoff...")
                time.sleep(10)
                ds = load_terraclimate_dataset()
                continue
            raise

Terraclimate_training_df = pd.concat(feature_dfs, axis=1)

100%|██████████| 60/60 [44:30<00:00, 44.51s/it]


Filtering for pet completed
pet: saved rows 0..199
pet: saved rows 200..399
pet: saved rows 400..599
pet: saved rows 600..799
pet: saved rows 800..999
pet: saved rows 1000..1199
pet: saved rows 1200..1399
pet: saved rows 1400..1599
pet: saved rows 1600..1799
pet: saved rows 1800..1999
pet: saved rows 2000..2199
pet: saved rows 2200..2399
pet: saved rows 2400..2599
pet: saved rows 2600..2799
pet: saved rows 2800..2999
pet: saved rows 3000..3199
pet: saved rows 3200..3399
pet: saved rows 3400..3599
pet: saved rows 3600..3799
pet: saved rows 3800..3999
pet: saved rows 4000..4199
pet: saved rows 4200..4399
pet: saved rows 4400..4599
pet: saved rows 4600..4799
pet: saved rows 4800..4999
pet: saved rows 5000..5199
pet: saved rows 5200..5399
pet: saved rows 5400..5599
pet: saved rows 5600..5799
pet: saved rows 5800..5999
pet: saved rows 6000..6199
pet: saved rows 6200..6399
pet: saved rows 6400..6599
pet: saved rows 6600..6799
pet: saved rows 6800..6999
pet: saved rows 7000..7199
pet: saved r

  0%|          | 0/60 [00:00<?, ?it/s]


Token expired while processing aet, refreshing dataset...


 15%|█▌        | 9/60 [07:07<40:24, 47.54s/it]


KeyboardInterrupt: 

In [8]:
Terraclimate_training_df['Latitude'] = Water_Quality_df['Latitude']
Terraclimate_training_df['Longitude'] = Water_Quality_df['Longitude']
Terraclimate_training_df['Sample Date'] = Water_Quality_df['Sample Date']

# Keep columns in requested order
Terraclimate_training_df = Terraclimate_training_df[
    ['Latitude', 'Longitude', 'Sample Date'] + vars_to_extract
]

# Save to repo root for comparison
Terraclimate_training_df.to_csv('test_train.csv', index=False)

In [9]:
# Preview File
Terraclimate_training_df.head()

,Latitude,Longitude,Sample Date,pet
0,-28.760833,17.730278,02-01-2011,174.199997
1,-26.861111,28.884722,03-01-2011,124.099998
2,-26.450000,28.085833,03-01-2011,127.500000
3,-27.671111,27.236944,03-01-2011,129.699997
4,-27.356667,27.286389,03-01-2011,129.199997


### Extracting features for the validation dataset

In [12]:
# Load submission template (repo root)
Validation_df = pd.read_csv('submission_template.csv')
Validation_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [13]:
Validation_df.shape

(200, 6)

In [ ]:
# Load TerraClimate dataset, filter (time,region,parameter), filter for nearest parameter values
feature_dfs = []
for var in vars_to_extract:
    while True:
        try:
            tc_parameter = filterg(ds, var)
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"val_{var}.csv")
            feature_dfs.append(assign_nearest_climate_chunked(Validation_df, tc_parameter, var, ckpt_path))
            break
        except Exception as e:
            err_text = str(e)
            if "AuthenticationFailed" in err_text or "Signature not valid" in err_text:
                print(f"Token expired while processing {var} (validation), refreshing dataset...")
                time.sleep(5)
                ds = load_terraclimate_dataset()
                continue
            if "Timeout" in err_text or "SocketTimeoutError" in err_text or "ServiceResponseTimeoutError" in err_text:
                print(f"Timeout while processing {var} (validation), retrying after backoff...")
                time.sleep(10)
                ds = load_terraclimate_dataset()
                continue
            raise

Terraclimate_validation_df = pd.concat(feature_dfs, axis=1)

 77%|███████▋  | 46/60 [42:01<12:47, 54.81s/it] 


Token expired while processing pet (validation), refreshing dataset...


  7%|▋         | 4/60 [03:46<51:12, 54.87s/it]  

In [13]:
Terraclimate_validation_df['Latitude'] = Validation_df['Latitude']
Terraclimate_validation_df['Longitude'] = Validation_df['Longitude']
Terraclimate_validation_df['Sample Date'] = Validation_df['Sample Date']

# Keep columns in requested order
Terraclimate_validation_df = Terraclimate_validation_df[
    ['Latitude', 'Longitude', 'Sample Date'] + vars_to_extract
]

# Save to repo root for comparison
Terraclimate_validation_df.to_csv('test_test.csv', index=False)

In [14]:
# Preview File
Terraclimate_validation_df.head()

,Latitude,Longitude,Sample Date,pet
0,-32.043333,27.822778,01-09-2014,161.900009
1,-33.329167,26.077500,16-09-2015,177.600006
2,-32.991639,27.640028,07-05-2015,158.400009
3,-34.096389,24.439167,07-02-2012,130.000000
4,-32.000556,28.581667,01-10-2014,152.500000
